# Exemples du cours CH4 — SGBD vectoriels avec **Chroma** 


**Prérequis conseillés**  
- Python 3.10+  
- `pip install chromadb sentence-transformers scikit-learn matplotlib`  


## 1- Installation & imports

### ⚙️ Préambule — Environnement (threads / tokenizers) 

In [ ]:
# Limitation du  nombre de threads (OMP/MKL) 
# et désactivation du parallélisme des tokenizers pour éviter 
# que le notebook monopolise la machine 
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
# Si nécessaire, décommentez pour installer (exécuter une seule fois)
# !pip install -U chromadb sentence-transformers scikit-learn matplotlib
# Bonus Qdrant :
# !pip install -U qdrant-client

import os
import json
from typing import List, Dict, Any

import numpy as np
import matplotlib.pyplot as plt

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector DB Chroma
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions 


# Évaluation
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

# Outil pratique pour afficher des structures Python de manière lisible
from pprint import pprint



## 2- Présentation des fonctionnalités sur un exemple simple  

***Intialiser Chroma***  
🗂️ Démarrage de Chroma en mode persistant et première requête filtrée  
Le code initialise un dossier local `chroma_data` pour stocker les données, crée une collection appelée **corpus_demo**, insère des documents avec leurs embeddings, puis exécute une requête restreinte à la catégorie *animaux*.  
La ligne `Settings(is_persistent=True, persist_directory=...)` active le **mode persistant** de Chroma : cela signifie que la base n’est plus uniquement stockée en mémoire mais sauvegardée sur disque dans le dossier `chroma_data`.  
Ce dossier contient les fichiers de configuration, les embeddings, les métadonnées et l’index FAISS. Concrètement, cela permet de conserver la collection entre deux exécutions du notebook, contrairement au mode mémoire où tout est perdu à la fermeture du kernel. 

In [68]:
# Client SGBD (persistance locale)
client = chromadb.Client(Settings(
    is_persistent=True,
    persist_directory="chroma_data"
))

In [79]:
# Afficher la liste des collections
print(client.list_collections())

[]


In [78]:
#Supprimer toutes les collections
for c in client.list_collections():
    client.delete_collection(name=c.name)
print("Toutes les collections ont été supprimées.")
#client.delete_collection(name="articles_demo")


Toutes les collections ont été supprimées.


***Définir un corpus exemple trés simple***

In [80]:
docs = [
    "Le chat chasse une souris.",           # animaux
    "Le chien aboie fort.",                 # animaux
    "Le poisson nage dans l’aquarium.",     # animaux
    "Le cuisinier prépare un gâteau.",      # cuisine
    "La soupe mijote dans la casserole."    # cuisine
]
metas = [
    {"cat": "animaux"},
    {"cat": "animaux"},
    {"cat": "animaux"},
    {"cat": "cuisine"},
    {"cat": "cuisine"}
]
ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]


***Créer une collection avec des embeddings pré-encodés***

In [71]:
# choix d'un modèle d'embedding
MODEL_NAME = "all-MiniLM-L6-v2"
#MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)
# encodage des documents
E = model.encode(docs, normalize_embeddings=True)
# creation de la collection
collection1 = client.get_or_create_collection(name="demo",
     metadata={"hnsw:space": "ip"}  # espace de similarité (cosine|l2|ip)
)
# ajout des documents
collection1.add(
    documents=docs,
    ids=ids,
    metadatas=metas,
    embeddings=E.tolist() ) 

snap = collection1.get()
print("Clés disponibles :", list(snap.keys()))
print("Documents :", snap)
print("Nombre de documents indexés :", len(snap["ids"]))



Clés disponibles : ['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas']
Documents : {'ids': ['doc1', 'doc2', 'doc3', 'doc4', 'doc5'], 'embeddings': None, 'documents': ['Le chat chasse une souris.', 'Le chien aboie fort.', 'Le poisson nage dans l’aquarium.', 'Le cuisinier prépare un gâteau.', 'La soupe mijote dans la casserole.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'cat': 'animaux'}, {'cat': 'animaux'}, {'cat': 'animaux'}, {'cat': 'cuisine'}, {'cat': 'cuisine'}]}
Nombre de documents indexés : 5


***Créer une collection avec encodage automatique dans Chroma***

In [72]:
# from chromadb.utils import embedding_functions 
# choix d'une fonction d'embedding
embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
# creation de la collection

collection = client.get_or_create_collection(name="demo_auto",embedding_function=embedder, 
    metadata={"hnsw:space": "ip"} ) # espace de similarité (cosine|l2|ip)
# ajout des documents
collection.add(
    documents=docs,
    ids=ids,
    metadatas=metas)
snap = collection.get()
print("Clés disponibles :", list(snap.keys()))
print("Documents :",  snap)
print("Nombre de documents indexés :", len(snap["ids"]))



Clés disponibles : ['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas']
Documents : {'ids': ['doc1', 'doc2', 'doc3', 'doc4', 'doc5'], 'embeddings': None, 'documents': ['Le chat chasse une souris.', 'Le chien aboie fort.', 'Le poisson nage dans l’aquarium.', 'Le cuisinier prépare un gâteau.', 'La soupe mijote dans la casserole.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'cat': 'animaux'}, {'cat': 'animaux'}, {'cat': 'animaux'}, {'cat': 'cuisine'}, {'cat': 'cuisine'}]}
Nombre de documents indexés : 5


***Mises à jour***

In [73]:
collection.add(
    documents=["le chat dort"],
    metadatas=[{"cat": "animaux"}],
    ids=["doc0"]
)

collection.update(
    ids=["doc1"], 
    documents=["Le chat chasse des souris."],
    metadatas=[{"cat": "animaux"}]
)
# Suppression
collection.delete(ids=["doc0"])

snap = collection.get()

print("Clés disponibles :", list(snap.keys()))
print("Documents :",  snap)

print(client.list_collections())


Clés disponibles : ['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas']
Documents : {'ids': ['doc1', 'doc2', 'doc3', 'doc4', 'doc5'], 'embeddings': None, 'documents': ['Le chat chasse des souris.', 'Le chien aboie fort.', 'Le poisson nage dans l’aquarium.', 'Le cuisinier prépare un gâteau.', 'La soupe mijote dans la casserole.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'cat': 'animaux'}, {'cat': 'animaux'}, {'cat': 'animaux'}, {'cat': 'cuisine'}, {'cat': 'cuisine'}]}
[Collection(name=demo), Collection(name=demo_auto)]


***Accès aux documents avec GET***

In [ ]:
# Tous les docuuments d'une collection
snap = collection.get()

print("Clés disponibles :", list(snap.keys()))
print("Documents :",  snap)

# Accès direct à un document
res = collection.get(
    ids=["doc1"],  # Identifiants précis
    include=["documents", "metadatas"]  
	# Choix des éléments à récupérer
)
print(res)
print("Document doc1 :", res["documents"])

# filtrage par métadonnées
res = collection.get(
    include=["documents"],
    where ={"cat": "animaux"}
)
print("Documents trouvés :", res["documents"])

print(res)

print("Documents trouvés :")
for doc in res["documents"]:
    print(f"-{doc}")


***Recherche vectorielle (par similarité)***

In [74]:
#collection= collection obtenue par encodage chroma
#collection1= collection obtenue par encodage externe avant insertion dans chroma


#exemple de requete avec query_texts
res = collection.query(
    query_texts=["chat chasse"],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)
print("Documents trouvés pour [chat chasse] encodage Chroma:")
for id, doc, meta, dist in zip(res["ids"][0],res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"{id} | {meta['cat']:<10} | {dist:.3f} |  {doc}")

res = collection1.query(
    query_texts=["chat chasse"],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)
print("Documents trouvés pour [chat chasse] encodage Externe:")
for id, doc, meta, dist in zip(res["ids"][0],res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"{id} | {meta['cat']:<10} | {dist:.3f} |  {doc}")


#exemple avec filtrage pour ["chat chasse"] avec where (encodage chroma)
res = collection.query(
    query_texts=["chat chasse"],
    n_results=3,
    where={"cat": "animaux"}
)

print("Documents trouvés filtrage pour [chat chasse] avecencodage chroma:")
for id, doc, meta, dist in zip(res["ids"][0],res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"{id} | {meta['cat']:<10} | {dist:.3f} |  {doc}")


# exemple requête vectorielle pure encodage chroma
q_vec = model.encode(["chat chasse"], normalize_embeddings=True)

res = collection.query(
    query_embeddings=q_vec,      
    # ✅ pas de query_texts
    n_results=3,
    include=["documents", "metadatas", "distances"]
)
print("Documents trouvés req vectorielle encodage chroma :")
for id, doc, meta, dist in zip(res["ids"][0],res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"{id} | {meta['cat']:<10} | {dist:.3f} |  {doc}")




Documents trouvés pour [chat chasse] encodage Chroma:
doc1 | animaux    | 0.338 |  Le chat chasse des souris.
doc5 | cuisine    | 0.744 |  La soupe mijote dans la casserole.
doc2 | animaux    | 0.757 |  Le chien aboie fort.
Documents trouvés pour [chat chasse] encodage Externe:
doc1 | animaux    | 0.368 |  Le chat chasse une souris.
doc5 | cuisine    | 0.744 |  La soupe mijote dans la casserole.
doc2 | animaux    | 0.757 |  Le chien aboie fort.
Documents trouvés filtrage pour [chat chasse] avecencodage chroma:
doc1 | animaux    | 0.338 |  Le chat chasse des souris.
doc2 | animaux    | 0.757 |  Le chien aboie fort.
doc3 | animaux    | 0.881 |  Le poisson nage dans l’aquarium.
Documents trouvés req vectorielle encodage chroma :
doc1 | animaux    | 0.338 |  Le chat chasse des souris.
doc5 | cuisine    | 0.744 |  La soupe mijote dans la casserole.
doc2 | animaux    | 0.757 |  Le chien aboie fort.


## 3- Exemple final du cours

In [81]:
# pip install chromadb sentence-transformers
# import chromadb
# from chromadb.config import Settings
# Creation des documents

# 1) Client SGBD (persistance locale)
# client = chromadb.Client(Settings(
    # is_persistent=True,
    # persist_directory="chroma_data"   # dossier de la base
# ))
# 2) Créer / récupérer une "collection" (la base vectorielle)
# Nom de la collection à réinitialiser
COLLECTION_NAME = "articles_demo"

# Supprimer d'abord la collection si elle existe déjà
try:
    client.delete_collection(COLLECTION_NAME)
    print(f"✅ Collection '{COLLECTION_NAME}' supprimée")
except Exception:
    print(f"ℹ️ Collection '{COLLECTION_NAME}' n'existait pas")
#Créer la "collection" 
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    # metadata={"distance_function": "cosine"}  # espace de similarité (cosine|l2|ip)
    metadata={"hnsw:space": "ip"}  # espace de similarité (cosine|l2|ip) #choix IP= inner product 
)
print("Collections existantes :", client.list_collections())



ℹ️ Collection 'articles_demo' n'existait pas
Collections existantes : [Collection(name=articles_demo)]


***Encodage : produire des embeddings***

In [82]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

docs_economie = [
    "La banque centrale relève ses taux directeurs.",
    "Le gouvernement présente un budget de rigueur.",
    "L'inflation ralentit selon les derniers chiffres.",
    "Les marchés financiers terminent en hausse.",
    "La banque annonce des bénéfices record cette année."
]
docs_animaux = [
    "Le chat dort sur le canapé du salon.",
    "Un félin se repose près de la fenêtre.",
    "Le chien court dans le jardin en aboyant.",
    "Le chat chasse une souris dans la cuisine.",
    "Un vétérinaire soigne un chiot malade."
]
docs_meteo = [
    "La météo annonce de la pluie demain.",
    "Un ciel bleu et ensoleillé est attendu cet après-midi.",
    "Le vent souffle fortement sur la côte.",
    "De la neige est prévue en montagne ce week-end.",
    "Un épisode de canicule est redouté la semaine prochaine."
]
docs_transport = [
    "Le conducteur accélère sur l'autoroute A6.",
    "La voiture roule vite sur l'autoroute.",
    "Le train arrive à l'heure sur le quai 2.",
    "Des embouteillages sont signalés en périphérie.",
    "Un avion décolle avec du retard à cause du brouillard."
]
docs_politique = [
    "Le président visite la capitale pour un sommet.",
    "Le ministre inaugure un nouveau musée.",
    "Le parlement débat d'une réforme constitutionnelle.",
    "La campagne électorale bat son plein dans le pays.",
    "L'opposition critique la politique étrangère du gouvernement."
]
docs_culture = [
    "Le professeur corrige les copies des étudiants.",
    "Une exposition d'art contemporain ouvre ses portes.",
    "Le festival de musique débute vendredi soir.",
    "Une bibliothéque municipale propose des ateliers d'écriture.",
    "Un écrivain présente son nouveau roman en librairie."
]

docs = docs_economie + docs_animaux + docs_meteo + docs_transport + docs_politique + docs_culture

cats = (
    ["économie"]*len(docs_economie) +
    ["animaux"]*len(docs_animaux) +
    ["météo"]*len(docs_meteo) +
    ["transport"]*len(docs_transport) +
    ["politique"]*len(docs_politique) +
    ["culture"]*len(docs_culture)
)
ids = [f"doc_{i:03d}" for i in range(len(docs))]
metas = [{"cat": c} for c in cats]

len(docs), len(metas), metas[0]
# Embeddings normalisés (utile pour cosinus)
E = model.encode(docs, normalize_embeddings=True)

***Insertion d'embeddings et affichage***

In [83]:
# 1) Insertion dans la collection (SGBD)
collection.add(
    documents=docs,
    ids=ids,
    metadatas=metas,
    embeddings=E.tolist()
)
# 2)Récupérer les documents stockés
# get() retourne un dictionnaire avec ids, documents, embeddings, metadatas
docs = collection.get()

# 3)Affichage des documents
print("Nb d’objets :", collection.count())
for doc_id, doc_text, meta in zip(docs["ids"], docs["documents"], docs["metadatas"]):
    print(f"ID: {doc_id}")
    print(f"Texte: {doc_text}")
    print(f"Métadonnées: {meta}")
    print("-" * 40)



Nb d’objets : 30
ID: doc_000
Texte: La banque centrale relève ses taux directeurs.
Métadonnées: {'cat': 'économie'}
----------------------------------------
ID: doc_001
Texte: Le gouvernement présente un budget de rigueur.
Métadonnées: {'cat': 'économie'}
----------------------------------------
ID: doc_002
Texte: L'inflation ralentit selon les derniers chiffres.
Métadonnées: {'cat': 'économie'}
----------------------------------------
ID: doc_003
Texte: Les marchés financiers terminent en hausse.
Métadonnées: {'cat': 'économie'}
----------------------------------------
ID: doc_004
Texte: La banque annonce des bénéfices record cette année.
Métadonnées: {'cat': 'économie'}
----------------------------------------
ID: doc_005
Texte: Le chat dort sur le canapé du salon.
Métadonnées: {'cat': 'animaux'}
----------------------------------------
ID: doc_006
Texte: Un félin se repose près de la fenêtre.
Métadonnées: {'cat': 'animaux'}
----------------------------------------
ID: doc_007
Texte:

***Requêtes Top-k***

In [84]:

# 4) Requête vectorielle (embedding calculé explicitement)
# q = "La banque centrale annonce"
# q="Hausse des taux d'intérêt"
q="Un félin qui fait la sieste"

q_emb = model.encode([q], normalize_embeddings=True)

res = collection.query(
    query_embeddings=q_emb.tolist(),
    n_results=3,
    include=["documents", "metadatas", "distances"]
    #contrôle ce qui est renvoyé (docs/ids/metas/embeddings/distances)
)
cats = [m["cat"] for m in res["metadatas"][0]]
print(cats)
print("Requête :", q)
for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print(f"{dist:.3f} | {meta['cat']:<8} | {doc}")


['météo', 'météo', 'animaux']
Requête : Un félin qui fait la sieste
0.594 | météo    | Un épisode de canicule est redouté la semaine prochaine.
0.675 | météo    | Un ciel bleu et ensoleillé est attendu cet après-midi.
0.729 | animaux  | Un félin se repose près de la fenêtre.


***Filtrage, mise à jour, suppression***

In [ ]:
# Filtrer par métadonnées
res = collection.query(
    query_embeddings=q_emb.tolist(),
    n_results=3,
    where={"cat": "animaux"},      # filtre par payload
    include=["documents","metadatas","distances"]
)
# Affichage des documents filtrés
print("affichage documents filtrés")
for doc_id, doc_text, meta in zip(res["ids"], res["documents"], res["metadatas"]):
    print(f"ID: {doc_id}")
    print(f"Texte: {doc_text}")
    print(f"Métadonnées: {meta}")
    print("-" * 40)

# Mise à jour (upsert) : réécrire un doc existant
#On remplace ""Le chat dort sur le canapé du salon." par "Le chat ronronne près du poêle."
collection.update(
    ids=["doc_1"], 
    documents=["Le chat ronronne près du poêle."],
    metadatas=[{"source":"blog","cat":"animaux"}]
)
# affichage pour vérification
print("Affichage aprés mise à jour")
docs = collection.get()
for doc_id, doc_text, meta in zip(docs["ids"], docs["documents"], docs["metadatas"]):
    print(f"ID: {doc_id}")
    print(f"Texte: {doc_text}")
    print(f"Métadonnées: {meta}")
    print("-" * 40)
# Suppression du doc 2
collection.delete(ids=["doc_2"])
# affichage pour vérification
print("Affichage aprés suppression")
docs = collection.get()
for doc_id, doc_text, meta in zip(docs["ids"], docs["documents"], docs["metadatas"]):
    print(f"ID: {doc_id}")
    print(f"Texte: {doc_text}")
    print(f"Métadonnées: {meta}")
    print("-" * 40)

***Evaluation***  
- **Precision@k** = proportion de résultats corrects parmi les k premiers  
  
- **Recall@k** = proportion des documents pertinents retrouvés dans les k premiers par rapport à tous les documents pertinents du corpus

In [ ]:
# Mini-évaluation P@k / R@k
#Jeu de requêtes annotées

eval_set = [
    # économie
    ("Hausse des taux d'intérêt", "économie"),
    ("Prévisions d'inflation et budget de l'État", "économie"),

    # animaux
    ("Un félin qui fait la sieste", "animaux"),
    ("Un chien qui joue dans le jardin", "animaux"),

    # météo
    ("Risque de pluie demain", "météo"),
    ("Alerte canicule la semaine prochaine", "météo"),

    # transport
    ("Trafic dense sur l'autoroute", "transport"),
    ("Retard d'un train sur le quai", "transport"),

    # politique
    ("Le président s'exprime au sommet", "politique"),
    ("Le parlement discute d'une réforme", "politique"),

    # culture
    ("Ouverture d'une exposition d'art", "culture"),
    ("Un festival de musique commence vendredi", "culture")
]
def precision_recall_at_k(results_cats, target_cat, k):
    #Fonction d’évaluation
    topk = results_cats[:k]                 # Catégories des k premiers résultats
    tp = sum(1 for c in topk if c == target_cat)   # Nombre de Vrais positifs (bons résultats)
    total_rel = metas.count({"cat": target_cat})  # Nombre total de documents pertinents
    precision = tp / k
    recall = tp / max(1, total_rel)         # max(1,…) pour éviter la division par zéro
    return precision, recall

# Boucle d’évaluation
K = 3
scores = []
for q, gold_cat in eval_set:
    #Chaque requête est transformée en embedding (q_emb)
    q_emb = model.encode([q], convert_to_numpy=True, normalize_embeddings=True)
    #On interroge Chroma pour récupérer les K=3 voisins les plus proches
    res = collection.query(query_embeddings=q_emb.tolist(), n_results=K, include=["metadatas"])
    #On extrait les catégories des résultats (cats)
    cats = [m["cat"] for m in res["metadatas"][0]]
    #On calcule précision & rappel pour cette requête
    p, r = precision_recall_at_k(cats, gold_cat, K)
    #On stocke les scores dans scores
    scores.append((q, gold_cat, p, r))

# Affichage détaillé du résultatprint("Precision/Recall@3")
print("Precision/Recall@3")
print(f"{'Query':<40} | {'Gold':<10} | {'P@3':>5} | {'R@3':>5}")
print("-" * 70)

for q, gold, p, r in scores:
    q_trunc = (q[:37] + '...') if len(q) > 40 else q  # coupe les requêtes trop longues
    print(f"{q_trunc:<40} | {gold:<10} | {p:5.2f} | {r:5.2f}")



